In [81]:
from ISLP import load_data
from ISLP.models import ModelSpec as MS, sklearn_sm, summarize, poly
import numpy as np
from sklearn.base import clone
import statsmodels.api as sm
from functools import partial
import pandas as pd

In [82]:
Portfolio = load_data('Portfolio')

def alpha_func(D,idx):
    cov_ = np.cov(D[['X','Y']].loc[idx],rowvar=False)
    return ((cov_[1,1]-cov_[0,1])/(cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

In [83]:
print(alpha_func(Portfolio,range(100)))

0.57583207459283


In [84]:
rng = np.random.default_rng(0)
print(alpha_func(Portfolio,rng.choice(100,100,replace=True)))

0.6074452469619004


In [85]:
def boot_SE(func,D,n=None,B=1000,seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0,0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,n,replace=True)
        value = func(D,idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

In [86]:
alpha_SE = boot_SE(alpha_func,Portfolio,B=1000,seed=0)

print(alpha_SE)

0.09118176521277699


In [87]:
def boot_OLS(model_matrix,response,D,idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_,X_).fit().params

In [88]:
hp_func = partial(boot_OLS, MS(['horsepower']),'mpg')

In [99]:
Auto = pd.read_csv('Auto.csv')
Auto = Auto[Auto['horsepower']!='?'].copy()
Auto['horsepower'] = Auto['horsepower'].astype(int)
Auto = Auto.reset_index()

rng = np.random.default_rng(0)
np.array([hp_func(Auto,rng.choice(392,392,replace=True)) for _ in range(10)])

array([[39.88064456, -0.1567849 ],
       [38.73298691, -0.14699495],
       [38.31734657, -0.14442683],
       [39.91446826, -0.15782234],
       [39.43349349, -0.15072702],
       [40.36629857, -0.15912217],
       [39.62334517, -0.15449117],
       [39.0580588 , -0.14952908],
       [38.66688437, -0.14521037],
       [39.64280792, -0.15555698]])

In [90]:
hp_se = boot_SE(hp_func,Auto,B=1000,seed=10)

display(hp_se)

intercept     0.848807
horsepower    0.007352
dtype: float64

In [94]:

hp_model = sklearn_sm(sm.OLS,MS(['horsepower']))

hp_model.fit(Auto,Auto['mpg'])

model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

In [96]:
quad_model = MS([poly('horsepower',2,raw=True)])

quad_func = partial(boot_OLS,quad_model,'mpg')
boot_SE(quad_func, Auto, B=1000)

intercept                                  2.067840
poly(horsepower, degree=2, raw=True)[0]    0.033019
poly(horsepower, degree=2, raw=True)[1]    0.000120
dtype: float64

In [97]:
M = sm.OLS(Auto['mpg'],quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64